In [1]:
import pandas as pd          # main data tool (like Excel in Python)
import numpy as np           # math & number operations
import matplotlib.pyplot as plt  # basic charts
import seaborn as sns        # nicer looking charts
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # show ALL columns
print('All libraries imported successfully!')

df = pd.read_csv('D:\Technocolabs\Pickup Five Cities Datasets\pickup_jl.csv')

print('Number of rows   :', df.shape[0])  # how many records
print('Number of columns:', df.shape[1])  # how many fields

All libraries imported successfully!
Number of rows   : 261801
Number of columns: 19


In [2]:
df.isnull().sum()

order_id                 0
region_id                0
city                     0
courier_id               0
accept_time              0
time_window_start        0
time_window_end          0
lng                      0
lat                      0
aoi_id                   0
aoi_type                 0
pickup_time              0
pickup_gps_time      42053
pickup_gps_lng       42053
pickup_gps_lat       42053
accept_gps_time      81832
accept_gps_lng       81832
accept_gps_lat       81832
ds                       0
dtype: int64

In [4]:
gps_cols = ['pickup_gps_lng', 'pickup_gps_lat',
            'accept_gps_lng',  'accept_gps_lat']
df[gps_cols] = df[gps_cols].fillna(0.0)

print(df[gps_cols].isnull().sum())

pickup_gps_lng    0
pickup_gps_lat    0
accept_gps_lng    0
accept_gps_lat    0
dtype: int64


In [3]:
# Step 1 — Reload the original file fresh (resets everything)
df = pd.read_csv('D:\Technocolabs\Pickup Five Cities Datasets\pickup_jl.csv')

# Step 2 — Convert datetime columns (format is MM-DD HH:MM:SS, add year manually)
time_cols = ['accept_time', 'time_window_start', 'time_window_end', 'pickup_time']

for col in time_cols:
    df[col] = pd.to_datetime('2023-' + df[col],
                              format='%Y-%m-%d %H:%M:%S',
                              errors='coerce')

# Step 3 — Verify it worked
print(df[time_cols].dtypes)
print()
print(df['accept_time'].head())

accept_time          datetime64[ns]
time_window_start    datetime64[ns]
time_window_end      datetime64[ns]
pickup_time          datetime64[ns]
dtype: object

0   2023-08-21 16:30:00
1   2023-09-24 15:24:00
2   2023-09-07 13:12:00
3   2023-10-04 13:50:00
4   2023-10-07 08:00:00
Name: accept_time, dtype: datetime64[ns]


In [9]:
duplicates = df.duplicated(subset=['order_id'])
print('Number of duplicate rows:', duplicates.sum())

Number of duplicate rows: 0


In [4]:
# Example: How many minutes from accept to pickup?
df['accept_to_pickup_min'] = (
    (df['pickup_time'] - df['accept_time']).dt.total_seconds() / 60
)
print(df['accept_to_pickup_min'].describe().round(1))
# This shows average, min, max pickup time in minutes

count    261801.0
mean        172.1
std         321.1
min           0.0
25%          45.0
50%          92.0
75%         160.0
max        8599.0
Name: accept_to_pickup_min, dtype: float64


In [5]:
col = 'accept_to_pickup_min'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = (df[col] < lower) | (df[col] > upper)
print(f'Outlier rows found: {outliers.sum():,}')
print(f'Valid range: {lower:.0f} to {upper:.0f} minutes')

Outlier rows found: 28,777
Valid range: -128 to 332 minutes


In [6]:
df = df[df['accept_to_pickup_min'] >= 0].reset_index(drop=True)
print('Rows after removing negatives:', len(df))

Rows after removing negatives: 261801


In [7]:
df['accept_to_pickup_min'] = df['accept_to_pickup_min'].clip(upper=upper)
print('Max duration after clipping:', df['accept_to_pickup_min'].max())

Max duration after clipping: 332.5


In [8]:
df['accept_hour']  = df['accept_time'].dt.hour    # 0 to 23
df['accept_day']   = df['accept_time'].dt.day     # 1 to 31
df['accept_month'] = df['accept_time'].dt.month   # 1 to 12
print(df[['accept_time', 'accept_hour', 'accept_day', 'accept_month']].head(3))

          accept_time  accept_hour  accept_day  accept_month
0 2023-08-21 16:30:00           16          21             8
1 2023-09-24 15:24:00           15          24             9
2 2023-09-07 13:12:00           13           7             9


In [9]:
def time_of_day(hour):
    if   0 <= hour < 6:   return 'Night'
    elif 6 <= hour < 12:  return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else:                  return 'Night'
df['time_of_day'] = df['accept_hour'].apply(time_of_day)
print(df['time_of_day'].value_counts())
print(df['time_of_day'].value_counts().sum())

time_of_day
Morning      201813
Afternoon     59715
Evening         256
Night            17
Name: count, dtype: int64
261801


In [17]:
df['window_duration_min'] = (
    (df['time_window_end'] - df['time_window_start']).dt.total_seconds() / 60
)
print(df['window_duration_min'].value_counts().head())


window_duration_min
120.0    243016
899.0     18145
714.0         5
634.0         5
817.0         5
Name: count, dtype: int64


In [10]:
print('Final dataset shape:', df.shape)
print()
print('Missing values left:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('New columns created:')
new_cols = ['has_pickup_gps', 'has_accept_gps', 'accept_to_pickup_min',
            'pickup_on_time', 'accept_hour', 'accept_day', 'accept_month',
            'time_of_day', 'window_duration_min']
print(new_cols)

Final dataset shape: (261801, 24)

Missing values left:
pickup_gps_time    42053
pickup_gps_lng     42053
pickup_gps_lat     42053
accept_gps_time    81832
accept_gps_lng     81832
accept_gps_lat     81832
dtype: int64

New columns created:
['has_pickup_gps', 'has_accept_gps', 'accept_to_pickup_min', 'pickup_on_time', 'accept_hour', 'accept_day', 'accept_month', 'time_of_day', 'window_duration_min']


In [12]:
# Check all text columns and their unique values
text_cols = df.select_dtypes(include='object').columns.tolist()
print('Text columns:', text_cols)
print()

for col in text_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()
    

Text columns: ['city', 'pickup_gps_time', 'accept_gps_time', 'time_of_day']

--- city ---
city
Jilin    261801
Name: count, dtype: int64

--- pickup_gps_time ---
pickup_gps_time
06-09 09:30:00    14
08-15 09:45:00    14
07-23 09:45:00    13
08-15 09:47:00    13
09-01 09:54:00    13
                  ..
07-17 17:21:00     1
06-01 11:27:00     1
07-10 12:02:00     1
09-05 16:58:00     1
05-30 09:25:00     1
Name: count, Length: 88664, dtype: int64

--- accept_gps_time ---
accept_gps_time
05-27 07:48:00    32
09-28 07:38:00    30
07-12 07:39:00    29
06-29 07:39:00    29
08-26 07:46:00    27
                  ..
05-26 13:53:00     1
05-14 12:18:00     1
06-18 10:29:00     1
05-28 13:10:00     1
09-26 11:19:00     1
Name: count, Length: 65224, dtype: int64

--- time_of_day ---
time_of_day
Morning      201813
Afternoon     59715
Evening         256
Night            17
Name: count, dtype: int64



In [13]:
for col in text_cols:
    df[col] = df[col].str.strip()        # remove spaces before/after
    df[col] = df[col].str.title()        # Capitalize First Letter
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)  # remove double spaces

print('Text columns cleaned!')
print(df['city'].value_counts())

Text columns cleaned!
city
Jilin    261801
Name: count, dtype: int64


In [14]:
# Cell 12b — Save cleaned data to a new CSV file
df.to_csv('pickup_jl_cleaned.csv', index=False)
print('File saved as: pickup_jl_cleaned.csv')
print(f'Total rows saved: {len(df):,}')

File saved as: pickup_jl_cleaned.csv
Total rows saved: 261,801
